
# Customer Churn Prediction Inference with MLflow

This notebook loads:
- the trained Keras model from MLflow
- the scaler and encoders from MLflow artifacts

Before running:
1. Make sure the training notebook has logged the model and artifacts
2. Set `MLFLOW_RUN_ID` to the run printed by the training notebook
3. Optionally set `MLFLOW_TRACKING_URI` for a remote MLflow server


In [5]:

import os
import pickle
import mlflow
import mlflow.keras
import pandas as pd
import numpy as np


In [6]:

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)

RUN_ID = os.getenv("MLFLOW_RUN_ID", "3bb1f5cfd4d345c7a1ac8e7cc6d31ffb")

MODEL_URI = os.getenv("MLFLOW_MODEL_URI", f"runs:/{RUN_ID}/model")
SCALER_URI = os.getenv("SCALER_URI", f"runs:/{RUN_ID}/scaler.pkl")
OHE_URI = os.getenv("OHE_URI", f"runs:/{RUN_ID}/onehot_encoder_geo.pkl")
GENDER_URI = os.getenv("GENDER_URI", f"runs:/{RUN_ID}/label_encoder_gender.pkl")

print("MODEL_URI:", MODEL_URI)
print("SCALER_URI:", SCALER_URI)
print("OHE_URI:", OHE_URI)
print("GENDER_URI:", GENDER_URI)


MODEL_URI: runs:/3bb1f5cfd4d345c7a1ac8e7cc6d31ffb/model
SCALER_URI: runs:/3bb1f5cfd4d345c7a1ac8e7cc6d31ffb/scaler.pkl
OHE_URI: runs:/3bb1f5cfd4d345c7a1ac8e7cc6d31ffb/onehot_encoder_geo.pkl
GENDER_URI: runs:/3bb1f5cfd4d345c7a1ac8e7cc6d31ffb/label_encoder_gender.pkl


In [7]:

def load_pickle_from_mlflow(artifact_uri: str):
    local_path = mlflow.artifacts.download_artifacts(artifact_uri=artifact_uri)
    with open(local_path, "rb") as f:
        return pickle.load(f)


In [8]:

model = mlflow.keras.load_model(MODEL_URI)
onehot_encoder_geo = load_pickle_from_mlflow(OHE_URI)
label_encoder_gender = load_pickle_from_mlflow(GENDER_URI)
scaler = load_pickle_from_mlflow(SCALER_URI)

print("MLflow model and artifacts loaded successfully")


E0000 00:00:1775248990.766621   10947 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/prasanna/annclassification/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


MLflow model and artifacts loaded successfully


In [9]:

input_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000
}


In [10]:

geo_encoded = onehot_encoder_geo.transform([[input_data["Geography"]]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(["Geography"])
)

input_df = pd.DataFrame([input_data])
input_df["Gender"] = label_encoder_gender.transform(input_df["Gender"])
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df


/home/prasanna/annclassification/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [11]:

input_scaled = scaler.transform(input_df)
prediction = model.predict(input_scaled, verbose=0)
prediction_proba = float(prediction[0][0])

print("Churn Probability:", round(prediction_proba, 4))
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")


Churn Probability: 0.029
The customer is not likely to churn.
